# Coil 20

In [1]:
import numpy as np

In [2]:
import coil20
mat, label = coil20.read_data(num_of_class=2,random_choose_class=True, random_shuffle=True) 
# by default, read all classes, no random shuffle
mat.shape, label.shape

((144, 1024), (144,))

In [3]:
unique, counts = np.unique(label, return_counts=True)
print(unique,counts)
print(type(label),type(mat))

[14 16] [72 72]
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [4]:
import sys
sys.path.insert(0, '../algorithm')
sys.path.insert(0, '../eval_metrics')
from SpectralClustering import spectral_clustering
from ac import ac
from nmi import nmi

In [5]:
n_clusters = 2

In [6]:
# Usage example, assuming 'mat' and 'label' are defined and 'n_clusters' is set
affinity = 'nearest_neighbors'
gamma = 0.7

# No need to compute class weights before calling the function
new_spc_labels = spectral_clustering(mat, label, n_clusters, affinity=affinity, gamma=gamma, n_neighbors=10)

In [7]:
accuracy = ac(label, new_spc_labels)
nmi_score = nmi(label, new_spc_labels)

print("Spectral Clustering Evaluation:")
print("Accuracy:", accuracy)
print("NMI Score:", nmi_score)

Spectral Clustering Evaluation:
Accuracy: 1.0
NMI Score: 1.0


# Cifar 10

In [8]:
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.optimizers import Adam
import pickle

def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

# Load CIFAR-10 dataset
cifar10 = unpickle('../../dataset/cifar-10-batches-py/test_batch')
images = cifar10[b'data'].reshape((-1, 3, 32, 32)).transpose(0, 2, 3, 1)
labels = np.array(cifar10[b'labels'])

In [9]:
unique, counts = np.unique(labels, return_counts=True)
print(unique,counts)
print(type(label),type(mat))

[0 1 2 3 4 5 6 7 8 9] [1000 1000 1000 1000 1000 1000 1000 1000 1000 1000]
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [10]:
n_clusters = 10

In [11]:
selected_classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

mask = np.isin(labels, selected_classes)
images = images[mask]
labels = labels[mask]

# Map the selected class labels to 0 and 1
label_map = {label: i for i, label in enumerate(selected_classes)}
labels = np.array([label_map[label] for label in labels])

In [12]:
# Load pre-trained VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
model = Model(inputs=base_model.input, outputs=base_model.output)

# Extract features from CIFAR-10 images
features = model.predict(preprocess_input(images))
features = features.reshape(features.shape[0], -1)

# Load pre-trained VGG16 model
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))

# Fine-tuning the CNN model
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Add new layers on top of the pre-trained model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dense(n_clusters, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=x)

# Compile the model
model.compile(optimizer=Adam(lr=0.0001), loss='sparse_categorical_crossentropy')
model.fit(images, labels, epochs=10, batch_size=32)
features_fine_tuned = model.predict(images)

D:\Anaconda\envs\CA\lib\site-packages\keras\optimizer_v2\optimizer_v2.py:356: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  "The `lr` argument is deprecated, use `learning_rate` instead.")


Epoch 1/10
313/313 [==============================] - 96s 305ms/step - loss: 2.1862
Epoch 2/10
313/313 [==============================] - 98s 312ms/step - loss: 1.3744
Epoch 3/10
313/313 [==============================] - 98s 312ms/step - loss: 1.0895
Epoch 4/10
313/313 [==============================] - 98s 313ms/step - loss: 0.8680
Epoch 5/10
313/313 [==============================] - 98s 312ms/step - loss: 0.6639
Epoch 6/10
313/313 [==============================] - 97s 311ms/step - loss: 0.5199
Epoch 7/10
313/313 [==============================] - 97s 311ms/step - loss: 0.4039
Epoch 8/10
313/313 [==============================] - 96s 308ms/step - loss: 0.2889
Epoch 9/10
313/313 [==============================] - 97s 310ms/step - loss: 0.2306
Epoch 10/10
313/313 [==============================] - 98s 312ms/step - loss: 0.1728


In [15]:
affinity = 'nearest_neighbors'
gamma = 0.7
new_spc_labels = spectral_clustering(features_fine_tuned, label, n_clusters, affinity=affinity, gamma=gamma, n_neighbors=25)

In [16]:
# Evaluate clustering performance
accuracy = ac(labels, new_spc_labels)
nmi_score = nmi(labels, new_spc_labels)

print("Spectral Clustering Evaluation:")
print("Accuracy:", accuracy)
print("NMI Score:", nmi_score)

Spectral Clustering Evaluation:
Accuracy: 0.7581
NMI Score: 0.7317090445728405


# Reuters21578

In [17]:
import reuters
mat, label = reuters.read_data(num_of_class=15, random_shuffle=True, random_choose_class=True)
# mat, label = reuters.read_data(num_of_class=15, random_choose_class=True)
# only read 3 classes (randomly choose 3 classes), no random shuffle
# in text data, different classes may have different number of samples, 
# so the number of samples returned may be different across different runs
print(mat.shape, label.shape)
print(np.unique(label, return_counts=True))

Reading reuters21578 dataset...
Counting the number of occurrences of each topic...
Checking documents with topics that occur less than 10 times...
Removing documents with topics that occur less than 10 times...
(552, 18933) (552,)
(array([ 9, 10, 14, 16, 17, 20, 21, 24, 25, 30, 34, 35, 38, 40, 41],
      dtype=uint8), array([110,  90,  53,  45,  44,  38,  37,  30,  27,  18,  15,  14,  11,
        10,  10], dtype=int64))


In [18]:
from sklearn.feature_selection import chi2, SelectKBest

# Assuming X is your feature matrix and y is your target labels
k = 1000  # Number of top features to select
selector = SelectKBest(chi2, k=k)
X_selected = selector.fit_transform(mat, label)

In [19]:
unique, counts = np.unique(label, return_counts=True)
print(unique,counts)
print(type(label),type(mat))

[ 9 10 14 16 17 20 21 24 25 30 34 35 38 40 41] [110  90  53  45  44  38  37  30  27  18  15  14  11  10  10]
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [20]:
n_clusters = 15

In [21]:
affinity = 'nearest_neighbors'
gamma = 0.7

# new_spc_labels = spectral_clustering(mat, label, class_weights_dict, n_clusters, affinity=affinity, gamma=gamma)
new_spc_labels = spectral_clustering(X_selected, label, n_clusters, affinity=affinity, gamma=gamma, n_neighbors=20)

In [22]:
accuracy = ac(label, new_spc_labels)
nmi_score = nmi(label, new_spc_labels)

print("Spectral Clustering Evaluation:")
print("Accuracy:", accuracy)
print("NMI Score:", nmi_score)

Spectral Clustering Evaluation:
Accuracy: 0.46195652173913043
NMI Score: 0.5102123554264045


# tdt2

In [36]:
import tdt2
mat, label = tdt2.read_data(num_of_class=20, random_choose_class=True, random_shuffle=False)
# only read 3 classes (randomly choose 3 classes), random shuffle
# in text data, different classes may have different number of samples,
# so the number of samples returned may be different across different runs
print(mat.shape, label.shape)
print(np.unique(label, return_counts=True))
print(label[:20])

(8299, 36771) (8299,)
(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 13, 14, 15, 16, 19, 22, 25, 26,
       28, 29, 30], dtype=uint8), array([1844, 1828, 1222,  811,  441,  407,  272,  238,  226,  141,  140,
        131,  123,  104,   74,   66,   65,   58,   56,   52], dtype=int64))
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [37]:
# Assuming X is your feature matrix and y is your target labels
k = 200  # Number of top features to select
selector = SelectKBest(chi2, k=k)
X_selected = selector.fit_transform(mat, label)

In [38]:
unique, counts = np.unique(label, return_counts=True)
print(unique,counts)
print(type(label),type(mat))

[ 1  2  3  4  5  6  7  8  9 13 14 15 16 19 22 25 26 28 29 30] [1844 1828 1222  811  441  407  272  238  226  141  140  131  123  104
   74   66   65   58   56   52]
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [39]:
n_clusters = 20

In [40]:
affinity = 'nearest_neighbors'
gamma = 0.7

# new_spc_labels = spectral_clustering(mat, label, class_weights_dict, n_clusters, affinity=affinity, gamma=gamma)
new_spc_labels = spectral_clustering(X_selected, label, n_clusters, affinity=affinity, gamma=gamma, n_neighbors=20)

In [41]:
accuracy = ac(label, new_spc_labels)
nmi_score = nmi(label, new_spc_labels)

print("Spectral Clustering Evaluation:")
print("Accuracy:", accuracy)
print("NMI Score:", nmi_score)

Spectral Clustering Evaluation:
Accuracy: 0.3922159296300759
NMI Score: 0.5827969906807291
